# L2: Classification, Regression & Forecasting with TabPFN

Predictions with TabPFN. I will use `TabPFNClassifier` to assess heart-disease risk, `TabPFNRegressor` to predict house prices and inspect the uncertainty intervals behind every regression output, before applying TabPFN to time-series forecasting.

## 1. Setup

Import the evaluation metrics, dataset loaders, and plotting helpers, then authenticate your `tabpfn_client` session to use `TabPFNClassifier` and `TabPFNRegressor`.

In [4]:
!pip install requests==2.32.5 tabpfn-common-utils==0.2.23
#!pip install tabpfn
!pip install tabpfn_client

In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
import shap

import os
import pickle

#os.environ["TABPFN_TOKEN"] ="*********************************"
#os.environ["HF_TOKEN"] ="*********************************"

#os.environ.get("TABPFN_TOKEN")
#os.environ.get("HF_TOKEN")

from tabpfn_client import init, TabPFNClassifier, TabPFNRegressor

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## 2. Classification: is this patient at risk?

Let's start with binary classification on clinical data:

- **The dataset:** 920 patients pooled from four hospital studies (Cleveland, Hungary, Switzerland, and Long Beach), split into 700 patients for training context and 220 held out for testing.
- **The target:** Binary (`1` = coronary heart disease, `0` = no disease).
- **Missing values:** Roughly two-thirds of the records have missing measurements (`NaN`). TabPFN handles missing values natively—no imputation needed.

In [6]:
data = pd.read_csv("./stroke_data.csv", sep = ',')
len(data)



X = data.drop('stroke', axis=1)
y= data['stroke']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

feature_names = list(X_train.columns)
cat_idx = pd.CategoricalIndex(feature_names)

print (X)

       sex   age  hypertension  heart_disease  ever_married  work_type  \
0      1.0  63.0             0              1             1          4   
1      1.0  42.0             0              1             1          4   
2      0.0  61.0             0              0             1          4   
3      1.0  41.0             1              0             1          3   
4      1.0  85.0             0              0             1          4   
...    ...   ...           ...            ...           ...        ...   
40905  1.0  38.0             0              0             0          4   
40906  0.0  53.0             0              0             1          4   
40907  1.0  32.0             0              0             1          2   
40908  1.0  42.0             0              0             1          3   
40909  1.0  35.0             0              0             0          4   

       Residence_type  avg_glucose_level   bmi  smoking_status  
0                   1             228.69  36.6

In [7]:
y_train.head(3)

,stroke
31519,0
3473,1
7695,1


The interface looks just like scikit-learn's, but the mechanics under the hood are different:

- **Pretrained foundation model:** `fit` does **no** parameter training—it simply prepares and uploads your training table so the model can use it as context.
- **In-context learning:** All model compute happens at `predict` time. The transformer reads the training rows and new test rows together to predict in a single forward pass.
- **Repeated inference:** Because the work happens at `predict` time, repeated requests re-encode the same training context. Lesson 4 shows how a KV-cache removes that extra cost.

No hyperparameter tuning or manual feature encoding is needed: TabPFN automatically detects categorical columns and works out of the box.

In [8]:
#!pip install --upgrade tabpfn-client

In [ ]:
clf = TabPFNClassifier()
clf.fit(X_train, y_train)

pickle.dump(clf, open("heartstroke.mdl", 'wb'))

#clf = pickle.load(open("heartstroke.mdl", 'rb'))

### Predict one patient first

Before scoring the whole test set, let's predict a single patient. `predict` returns the binary class label (`0` or `1`), while `predict_proba` returns the calibrated probability behind that decision—which is the number a clinician would actually use to assess risk.

In [ ]:
patient = X_test.iloc[[0]]
display(patient)

In [ ]:
prediction = clf.predict(patient)[0]


print(f"predicted: {prediction:.0f}   "
      f"actual: {y_test.iloc[0]:.0f}")

In [ ]:
proba = clf.predict_proba(patient)[0]
print(f"P(healthy: label=0) = {proba[0]:.2f}   "
      f"P(disease: label=1) = {proba[1]:.2f}")

### Evaluate on all 220 held-out patients

One patient tells a story; 220 patients tell us how well the model generalizes. Instead of compressing performance into a single accuracy score, let's inspect the **confusion matrix**: each row is the patient's true status, and each column is the model's prediction. The diagonal cells show correct predictions, while the off-diagonal cells show the two types of errors—which carry very different costs in a clinical setting.

In [ ]:
predictions = clf.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    predictions,
    display_labels=["no disease", "disease"],
    cmap="Blues",
    colorbar=False,
);


Two metrics summarize those two types of errors:

- **Precision:** Of the patients the model flags as having heart disease, what fraction actually have it? Low precision means more false alarms and unnecessary follow-up tests.
- **Recall:** Of the patients who truly have heart disease, what fraction does the model catch? Low recall means missed diagnoses—the costlier mistake in a clinic.

In [ ]:
precision = precision_score(y_test, predictions)
recall = recall_score(y_test, predictions)

print(f"precision: {precision:.2f}")
print(f"recall:    {recall:.2f}")

## 3. Fit the cached model


Interpretability methods like SHAP and partial dependence evaluate the model hundreds of times on perturbed inputs. Fitting `TabPFNClassifier` with `fit_mode="fit_with_cache"` caches the training context on the server so repeated predictions don't re-encode the training table on every call.

In [ ]:
clf_c = TabPFNClassifier(
    fit_mode="fit_with_cache",
    #categorical_features_indices=cat_idx,
)
#quiet_client_call(
clf_c.fit (X_train, y_train )
#)


#quiet_client_ca

X_test1, X_test2, y_test1, y_test2 = train_test_split(
    X_test, y_test, test_size=0.33, random_state=42)

#quiet_client_call(
proba_test_c1 = clf_c.predict_proba( X_test1)[:, 1]
print(f"test AUC of the first half of data {roc_auc_score(y_test1, proba_test_c1):.3f}")

proba_test_c2 = clf_c.predict_proba( X_test2)[:, 1]
print(f"test AUC of the second half of data {roc_auc_score(y_test2, proba_test_c2):.3f}")

## 4. Compute SHAP values


Build a permutation SHAP explainer using a 10-patient background sample from the training set, and compute feature attributions for the test patients. Each patient's SHAP values measure how much each clinical feature shifts their predicted heart-disease probability away from the baseline average.

In [ ]:

X100 = shap.utils.sample(X, 100)  # 100 instances for use as the background distribution

explainer = shap.Explainer(clf.predict, X100)
shap_values = explainer(X)

# make a standard partial dependence plot
sample_ind = 20
shap.partial_dependence_plot(
    data[[1]],
    clf_c.predict,
    X100,
    model_expected_value=True,
    feature_expected_value=True,
    ice=False,
    shap_values=shap_values[sample_ind : sample_ind + 1, :],
)



### Explaining single predictions

Because SHAP values are additive, summing a patient's feature attributions with the base value recovers their exact predicted probability. Inspect two individual cases—the highest-risk patient and a borderline patient near the 50% threshold—using waterfall plots to see which measurements push their predicted risk up or down.

In [ ]:
predicted = shap_values.values.sum(axis=1) + shap_values.base_values
idx_high = int(np.argmax(predicted))
idx_borderline = int(np.argmin(np.abs(predicted - 0.5)))

In [ ]:
plot_patient_waterfall(
    shap_values, idx_high, X_explain, feature_names, "Highest-risk patient",
)

In [ ]:
plot_patient_waterfall(
    shap_values, idx_borderline, X_explain, feature_names,
    "Borderline patient",
)

## 5. Feature importance

Aggregate the SHAP values across all explained patients in a beeswarm plot. Features are ranked top-to-bottom by average absolute impact: each dot is one patient, its horizontal position shows whether that measurement pushed risk toward or away from heart disease, and its color shows whether the underlying feature value was high (red) or low (blue).

Aggregate the SHAP values across all explained patients in a beeswarm plot. Features are ranked top-to-bottom by average absolute impact: each dot is one patient, its horizontal position shows whether that measurement pushed risk toward or away from heart disease, and its color shows whether the underlying feature value was high (red) or low (blue).

In [ ]:
plot_beeswarm(shap_values, len(feature_names))

## 6. Partial dependence plots

While SHAP shows how much each feature contributed to specific predictions, partial dependence plots (PDPs) show the shape of the model's response across a feature's range. Sweep the four most impactful continuous features and plot their average predicted risk on a shared y-axis so effect sizes are directly comparable.

In [ ]:
top_continuous = top_continuous_features(
    shap_values, feature_names, CONTINUOUS)

plot_partial_dependence(
    clf, X_train.values, top_continuous, feature_names,
)

## Recap

In this lesson, you explored three methods to interpret TabPFN predictions:

- **Individual predictions (SHAP waterfall plots):** Explain a single patient's prediction by seeing how each feature pushes their risk up or down from the baseline average.
- **Global feature importance (SHAP beeswarm plot):** Plot all SHAP values together to get a clear picture of which features have the biggest overall impact on the model's predictions.
- **Feature response curves (Partial dependence plots):** See how the model's average predicted risk responds as you vary individual continuous feature values across their range.
- **Associations, not causality:** Keep in mind that these explanations reflect statistical associations and model sensitivities in the data, not causal relationships.